In [43]:
import yfinance as yf
import datetime

In [ ]:
aapl = yf.Ticker("AAPL")
aapl.info

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

In [44]:
attributs = ['open', 'dayHigh', 'dayLow', 'previousClose', 'volume']
text = aapl.get_info()
get_info = {key: text[key] for key in attributs}
get_info

{'open': 248.11,
 'dayHigh': 249.1999,
 'dayLow': 246.0,
 'previousClose': 248.96,
 'volume': 88331081}

In [45]:
other_at = aapl.analyst_price_targets
get_info.update(other_at)
get_info

{'open': 248.11,
 'dayHigh': 249.1999,
 'dayLow': 246.0,
 'previousClose': 248.96,
 'volume': 88331081,
 'current': 247.99,
 'high': 350.0,
 'low': 205.0,
 'mean': 295.43536,
 'median': 300.0}

In [16]:
aapl.calendar

{'Dividend Date': datetime.date(2026, 2, 12),
 'Ex-Dividend Date': datetime.date(2026, 2, 9),
 'Earnings Date': [datetime.date(2026, 4, 30)],
 'Earnings High': 2.16,
 'Earnings Low': 1.85,
 'Earnings Average': 1.95539,
 'Revenue High': 112596000000,
 'Revenue Low': 105000000000,
 'Revenue Average': 109084922300}

In [55]:
import sys
from pathlib import Path

_root = Path.cwd()
if (_root / "pyproject.toml").exists():
    project_root = _root
else:
    # Notebook is under src/ingestion — go up to repo root
    project_root = Path.cwd().resolve().parent.parent

sys.path.insert(0, str(project_root))

from src.utils.logger import get_logger

# Initialize logger
logger = get_logger(__name__)

In [ ]:
def parse_info(ticker: str | yf.Ticker) -> dict:
    """Build a payload from yfinance ticker info and analyst price targets."""
    ticker_obj = ticker if isinstance(ticker, yf.Ticker) else yf.Ticker(ticker.strip())

    try:
        ticker_info = ticker_obj.info
        if not ticker_info:
            raise ValueError("Empty ticker info (invalid symbol or no data)")

        ta = ticker_obj.analyst_price_targets
        if ta is None:
            ticker_analysis = {}
        elif hasattr(ta, "to_dict"):
            ticker_analysis = ta.to_dict()
        else:
            ticker_analysis = dict(ta) if isinstance(ta, dict) else {}

        payload = {
            "ticker": ticker_info.get("symbol"),
            "company_name": ticker_info.get("longName"),
            "source_type": "yfinance (api)",
            "metrics": {
                "open": ticker_info.get("open"),
                "current_price": ticker_info.get("currentPrice"),
                "dayHigh": ticker_info.get("dayHigh"),
                "dayLow": ticker_info.get("dayLow"),
                "previousClose": ticker_info.get("previousClose"),
                "volume": ticker_info.get("volume"),
                "Global Metrics": {
                    "High Price": ticker_analysis.get("high"),
                    "Low Price": ticker_analysis.get("low"),
                    "Average Price": ticker_analysis.get("mean"),
                    "Median Price": ticker_analysis.get("median"),
                },
            },
            "metadata": {
                "status": "success",
                "message": "Ticker info fetched successfully",
            },
        }

        logger.info("Payload successfully created for %s", ticker_obj.ticker)
        return payload

    except Exception as e:
        logger.exception("Payload creation failed: %s", e)
        return {
            "ticker": getattr(ticker_obj, "ticker", None),
            "source_type": "yfinance (api)",
            "metadata": {
                "status": "error",
                "message": str(e),
            },
        }

In [70]:
parse_info(aapl)

{'ticker': 'AAPL',
 'company_name': 'Apple Inc.',
 'source_type': 'yfinance (api)',
 'metrics': {'open': 248.11,
  'current_price': 247.99,
  'dayHigh': 249.1999,
  'dayLow': 246.0,
  'previousClose': 248.96,
  'volume': 88331081,
  'Global Metrics': {'High Price': 350.0,
   'Low Price': 205.0,
   'Average Price': 295.43536,
   'Median Price': 300.0}},
 'metadata': {'status': 'success',
  'message': 'Ticker info fetched successfully'}}

In [76]:
ticket_list = ["AAPL", "MSFT", "TSLA"]

In [78]:
parse_list = { f"{ticket}":parse_info(ticket) for ticket in ticket_list}
print(parse_list)

{'AAPL': {'ticker': 'AAPL', 'company_name': 'Apple Inc.', 'source_type': 'yfinance (api)', 'metrics': {'open': 248.11, 'current_price': 247.99, 'dayHigh': 249.1999, 'dayLow': 246.0, 'previousClose': 248.96, 'volume': 88331081, 'Global Metrics': {'High Price': 350.0, 'Low Price': 205.0, 'Average Price': 295.43536, 'Median Price': 300.0}}, 'metadata': {'status': 'success', 'message': 'Ticker info fetched successfully'}}, 'MSFT': {'ticker': 'MSFT', 'company_name': 'Microsoft Corporation', 'source_type': 'yfinance (api)', 'metrics': {'open': 386.26, 'current_price': 381.85, 'dayHigh': 387.0, 'dayLow': 380.12, 'previousClose': 389.04, 'volume': 50853154, 'Global Metrics': {'High Price': 730.0, 'Low Price': 392.0, 'Average Price': 594.6217, 'Median Price': 600.0}}, 'metadata': {'status': 'success', 'message': 'Ticker info fetched successfully'}}, 'TSLA': {'ticker': 'TSLA', 'company_name': 'Tesla, Inc.', 'source_type': 'yfinance (api)', 'metrics': {'open': 379.39, 'current_price': 367.96, 'da